In [2]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns
from pathlib import Path 
 
ROOT=Path.cwd().parent
df=pd.read_csv(ROOT/'data'/'raw'/'diabetic_data.csv')

print(f'Shape: {df.shape}')
print(df.dtypes)
print(df.isnull().sum().sort_values(ascending=False).head(20))

print('\nReadmission distribution:')
print(df['readmitted'].value_counts())
print(df['readmitted'].value_counts(normalize=True).round(3)*100)

Shape: (101766, 50)
encounter_id                 int64
patient_nbr                  int64
race                        object
gender                      object
age                         object
weight                      object
admission_type_id            int64
discharge_disposition_id     int64
admission_source_id          int64
time_in_hospital             int64
payer_code                  object
medical_specialty           object
num_lab_procedures           int64
num_procedures               int64
num_medications              int64
number_outpatient            int64
number_emergency             int64
number_inpatient             int64
diag_1                      object
diag_2                      object
diag_3                      object
number_diagnoses             int64
max_glu_serum               object
A1Cresult                   object
metformin                   object
repaglinide                 object
nateglinide                 object
chlorpropamide              object


In [3]:
#replacing missing values
df=df.replace('?',np.nan)

#Check missing rates
missing=df.isnull().sum()/len(df)*100
print('Columns with >10% missing:')
print(missing[missing>10].sort_values(ascending=False))

Columns with >10% missing:
weight               96.858479
max_glu_serum        94.746772
A1Cresult            83.277322
medical_specialty    49.082208
payer_code           39.557416
dtype: float64


In [4]:
df=df.drop(columns=['weight','medical_specialty','payer_code'])
df=df.dropna(subset=['race'])
print(f'Shape after cleaning: {df.shape}')

Shape after cleaning: (99493, 47)


In [5]:
#'<30' means readmitted within 30 days
df['readmitted_30']=(df['readmitted']=='<30').astype(int)
print('30-day readmission rates:')
print(f"{df['readmitted_30'].mean()*100:.2f}%")
print(f"Count: {df['readmitted_30'].sum():,} out of {len(df):,}")

30-day readmission rates:
11.23%
Count: 11,169 out of 99,493


In [7]:
#Converting age brackets into numeric breakpoints
age_map={
    '[0-10)':10, '[10-20)':15, '[20-30)':25, '[30-40)':35,
    '[40-50)':45, '[50-60)':55, '[60-70)':65, '[70-80)':75,
    '[80-90)':85, '[90-100)':95
}
df['age_numeric']=df['age'].map(age_map)

#Mapping admission type to readable labels
admission_map={
    1:"Emergency", 2:'Urgent',3:'Elective', 4:'Newborn',5:'Not Available',
    6: 'NULL', 7:'Trauma Center', 8: 'Not Mapped'
}
df['admission_type']=df['admission_type_id'].map(admission_map)

df['prior_utilisation']=(df['number_outpatient']+df['number_emergency']+df['number_inpatient'])

#High complexity flag
df['high_complexity']=(
    (df['number_diagnoses']>=7) & 
    (df['num_medications']>=15)
).astype(int)

def classigy_diag(code):
    try:
        code=str(code)
        if code.startswith('V') or code.startswith('E'):
            return 'Other'
        n=float(code)
        if 390 <= n <= 459 or n==785: return 'Circulatory'
        elif 460 <= n <= 519 or n == 786: return 'Respiratory'
        elif 520 <= n <= 579 or n == 787: return 'Digestive'
        elif 800 <= n <= 999: return 'Injury'
        elif 710 <= n <= 739: return 'Musculoskeletal'
        elif 580 <= n <= 629 or n == 788: return 'Genitourinary'
        elif 140 <= n <= 239: return 'Neoplasms'
        elif n == 250: return 'Diabetes'
        else: return 'Other'
    except: return 'Other'

df['diag_category']=df['diag_1'].apply(classigy_diag)

#Save cleaned data
df.to_csv(ROOT/'data'/'processed'/'hospital_clean.csv',index=False)
print(f'Saved hospital cleaned data - {df.shape[0]:,} rows x {df.shape[1]} cols')

Saved hospital cleaned data - 99,493 rows x 53 cols
